In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import pypsa

/home/lukas/micromamba/envs/pypsa-eur/lib/python3.12/site-packages/google/cloud/storage/__init__.py:35: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution
ERROR 1: PROJ: proj_create_from_database: Open of /home/lukas/micromamba/envs/pypsa-eur/share/proj failed


In [51]:
n1 = pypsa.Network(
    Path.cwd().parent / 'results' / 'networks' / 'base_s_50__168H-T-H-B-I-A-dist1_2030_NT_-0.05.nc'
)
n2 = pypsa.Network(
    Path.cwd().parent / 'results' / 'networks' / 'base_s_50__168H-T-H-B-I-A-dist1_2030_NT_-0.05_50.nc'
)

INFO:pypsa.io:Imported network base_s_50__168H-T-H-B-I-A-dist1_2030_NT_-0.05.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores
INFO:pypsa.io:Imported network base_s_50__168H-T-H-B-I-A-dist1_2030_NT_-0.05_50.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores


In [52]:
n1.statistics()['Capital Expenditure'].sum()

430928942092.7436

In [53]:
n2.statistics()['Capital Expenditure'].sum()

431217094502.1702

In [49]:
n1.statistics()['Operational Expenditure'].sum()

492090165741.62964

In [50]:
n2.statistics()['Operational Expenditure'].sum()

589108349735.7537

In [ ]:
solver_kwargs = {
    "threads": 32,
    "method": 2, # barrier
    "crossover": 0,
    "BarConvTol": 1.e-5,
    "Seed": 123,
    "AggFill": 0,
    "PreDual": 0,
    "GURO_PAR_BARDENSETHRESH": 200
}

In [ ]:
def remove_flexibility_options(n, current_year):
    print("Removing decentral TES and BEV DSM from the network.")
    n.remove("Store", n.stores.query("carrier == 'EV battery'").index)
    carriers_to_drop = [
        "urban decentral water tanks charger",
        "urban decentral water tanks discharger",
        "urban decentral water tanks",
        "rural water tanks charger",
        "rural water tanks discharger",
        "rural water tanks",
    ]
    n.remove("Link", n.links.query(f"carrier in {carriers_to_drop}").index)
    n.remove("Store", n.stores.query(f"carrier in {carriers_to_drop}").index)
    n.remove("Bus", n.buses.query(f"carrier in {carriers_to_drop}").index)

    if current_year == 2030:
        print("Removing decentral TES and batteries from the network.")
        carriers_to_drop = [
            "home battery charger",
            "home battery discharger",
            "home battery",
            "battery charger",
            "battery discharger",
            "battery",
        ]
        n.remove(
            "Link",
            n.links.query(
                f"carrier in {carriers_to_drop} and build_year == {current_year}"
            ).index,
        )
        n.remove(
            "Store",
            n.stores.query(
                f"carrier in {carriers_to_drop} and build_year == {current_year}"
            ).index,
        )


def _unfix_bottlenecks(new, deci, name, extendable_i):
    if name == "links":
        # Links that have 0-cost and are extendable
        virtual_links = [
            "land transport oil",
            "land transport fuel cell",
            "solid biomass for industry",
            "gas for industry",
            "industry methanol",
            "naphtha for industry",
            "process emissions",
            "coal for industry",
            "H2 for industry",
            "shipping methanol",
            "shipping oil",
            "kerosene for aviation",
            "agriculture machinery oil",
            "co2 sequestered",
        ]

        _idx = new.loc[new.carrier.isin(virtual_links)].index.intersection(extendable_i)
        new.loc[_idx, "p_nom_extendable"] = True

        # Bottleneck links can be extended, but not reduced to fix infeasibilities due to numerical inconsistencies
        bottleneck_links = [
            "electricity distribution grid",
            "HVC to air",  # waste CHP would get used as a flexible energy source otherwise
            "SMR",
            # Boilers create bottlenecks AND should be extendable for fixed_profile_scaling constraints to be applied correctly
            "rural gas boiler",
            "urban decentral gas boiler",
            # Biomass for 2035 when gas is banned
            "rural biomass boiler",
            "urban decentral biomass boiler",
        ]
        _idx = new.loc[new.carrier.isin(bottleneck_links)].index.intersection(
            extendable_i
        )
        new.loc[_idx, "p_nom_extendable"] = True
        new.loc[_idx, "p_nom_min"] = deci.loc[_idx, "p_nom_opt"]
        # OCGT as last resort to avoid load shedding
        # allowed only in DE
        # (previously the model sometimes expanded waste CHPs)
        _idx = new.loc[
            (new.carrier == "OCGT") & (new.index.str.startswith("DE"))
        ].index.intersection(extendable_i)
        new.loc[_idx, "p_nom_extendable"] = True
        new.loc[_idx, "p_nom_min"] = deci.loc[_idx, "p_nom_opt"]

    if name == "generators":
        fuels = [
            "lignite",
            "coal",
            "oil primary",
            "uranium",
            "gas primary",
        ]
        vents = [
            "urban central heat vent",
            "rural heat vent",
            "urban decentral heat vent",
        ]
        _idx = new.loc[new.carrier.isin(fuels + vents)].index.intersection(extendable_i)
        new.loc[_idx, "p_nom_extendable"] = True

    return

In [17]:
n.links.carrier.unique()

array(['DC', 'co2 sequestered', 'OCGT', 'CCGT', 'H2 Electrolysis',
       'gas pipeline', 'gas pipeline new', 'battery charger',
       'battery discharger', 'Sabatier', 'SMR CC', 'SMR', 'BEV charger',
       'V2G', 'oil refining', 'land transport oil',
       'urban central air heat pump', 'urban central gas boiler',
       'urban central gas CHP', 'urban central gas CHP CC',
       'unsustainable bioliquids', 'biogas to gas',
       'urban central solid biomass CHP',
       'urban central solid biomass CHP CC', 'biomass to liquid',
       'electrobiofuels', 'Haber-Bosch', 'ammonia cracker',
       'biomass-to-methanol', 'OCGT methanol',
       'heat100-200 industry solid biomass',
       'heat100-200 industry solid biomass CC',
       'heat100-200 industry industrial heat pump high temperature',
       'heat100-200 industry electric boiler steam',
       'heat200-500 industry solid biomass',
       'heat200-500 industry solid biomass CC',
       'heat200-500 industry hydrogen', 'heat

In [30]:
nominal_attrs = {
    "generators": "p_nom",
    "lines": "s_nom",
    "links": "p_nom",
    "stores": "e_nom",
}

def fix_capacities(n_lt, no_flex=False):
    n = n_lt.copy()

    for name, attr in nominal_attrs.items():
        new = getattr(n, name)
        lt = getattr(n_lt, name)

        extendable_i = new.query(f"{attr}_extendable").index

        new.loc[extendable_i, attr + "_extendable"] = False
        new.loc[extendable_i, attr] = new.loc[extendable_i, attr + "_opt"]

        _unfix_bottlenecks(new, lt, name, extendable_i)

        # The CO2 constraints on atmosphere and sequestration need extendable stores to work correctly
        if name == "stores":
            print("Freeing co2 atmosphere and sequestered stores.")
            # there is only one co2 atmosphere store which should always be extendable, hence no intersection with extendable_i needed
            _idx = new.query("carrier == 'co2'").index
            new.loc[_idx, "e_nom_extendable"] = True
            # co2 sequestered stores from previous planning horizons should not be extendable
            _idx = new.query("carrier == 'co2 sequestered'").index.intersection(
                extendable_i
            )
            new.loc[_idx, "e_nom_extendable"] = True

        # Above several assets are switched to extendable again, for these the p_nom value is restored to the value from the decision network

        _idx = new.query(f"{attr}_extendable").index

        new.loc[_idx, attr] = lt.loc[_idx, attr]

    if no_flex:
        print("Realization network is from a run without flexibility.")
        remove_flexibility_options(n)
    return n

In [ ]:
n = n1.copy()
n = fix_capacities(n)

In [35]:
n.links.carrier.unique()

array(['DC', 'co2 sequestered', 'OCGT', 'CCGT', 'H2 Electrolysis',
       'gas pipeline', 'gas pipeline new', 'battery charger',
       'battery discharger', 'Sabatier', 'SMR CC', 'SMR', 'BEV charger',
       'V2G', 'oil refining', 'land transport oil',
       'urban central air heat pump', 'urban central gas boiler',
       'urban central gas CHP', 'urban central gas CHP CC',
       'unsustainable bioliquids', 'biogas to gas',
       'urban central solid biomass CHP',
       'urban central solid biomass CHP CC', 'biomass to liquid',
       'electrobiofuels', 'Haber-Bosch', 'ammonia cracker',
       'biomass-to-methanol', 'OCGT methanol',
       'heat100-200 industry solid biomass',
       'heat100-200 industry solid biomass CC',
       'heat100-200 industry industrial heat pump high temperature',
       'heat100-200 industry electric boiler steam',
       'heat200-500 industry solid biomass',
       'heat200-500 industry solid biomass CC',
       'heat200-500 industry hydrogen', 'heat

In [31]:
for c in ['Link']:

    df = n.components[c].static
    mask = (~df.p_nom_extendable) & (df.p_set == 0.) & (df.p_nom != 0.)
    df.loc[mask, 'p_set'] = np.nan

In [33]:
n.optimize(solver_name='gurobi')

INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 13/13 [00:00<00:00, 28.76it/s]
INFO:linopy.io: Writing time: 3.71s


Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2762156


INFO:gurobipy:Set parameter LicenseID to value 2762156


...

Solved in 0 iterations and 0.17 seconds (0.14 work units)


INFO:gurobipy:Solved in 0 iterations and 0.17 seconds (0.14 work units)


Infeasible or unbounded model


INFO:gurobipy:Infeasible or unbounded model
INFO:linopy.solvers:Unable to save solution file. Raised error: Unable to retrieve attribute 'X'
Status: warning
Termination condition: infeasible_or_unbounded
Solution: 0 primals, 0 duals
Objective: nan
Solver model: available
Solver message: 4



('warning', 'infeasible_or_unbounded')

In [28]:
diff = (n1.links['p_nom_opt'] - n2.links['p_nom']).abs().sort_values()
diff.tail(50)

Link
DE0 0 electricity distribution grid-reversed    14997.737850
IT0 0 electricity distribution grid             15147.381198
IT0 0 electricity distribution grid-reversed    15147.381198
FR0 0 land transport oil                        15168.655880
IT0 0 land transport oil                        15288.601087
BE0 0 urban decentral gas boiler                15348.650527
FR0 0 HVC to air                                16324.607083
FR0 0 electricity distribution grid             16337.201074
FR0 0 electricity distribution grid-reversed    16337.201074
IT0 0 HVC to air                                16483.583189
IT0 0 urban decentral gas boiler                16616.669551
GB2 0 electricity distribution grid-reversed    16815.778408
GB2 0 electricity distribution grid             16815.778408
DE0 2 electricity distribution grid-reversed    16945.510640
DE0 2 electricity distribution grid             16945.510640
NL0 0 electricity distribution grid-reversed    17102.032746
NL0 0 electricity d

In [38]:
n1.components['Link'].static

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,reversed,location,project_status,under_construction,length_original,tags,voltage,dc,underwater_fraction,geometry
Link,,,,,,,,,,,,,,,,,,,,,
relation/10377412-320-DC,FR0 3,GB2 1,,DC,0.969812,True,0,inf,1000.0,0.0,...,False,,,0.0,449.123466,relation/10377412,320.0,1.0,0.888446,LINESTRING (-1.194439480272105 50.818035638860...
relation/13295785-515-DC,NO1 0,GB2 0,,DC,0.956596,True,0,inf,1400.0,0.0,...,False,,,0.0,1038.803320,relation/13295785,515.0,1.0,0.983350,LINESTRING (-1.5404269162550226 55.14647596191...
relation/14126301-450-DC,GB2 1,NL0 0,,DC,0.969942,True,0,inf,1000.0,0.0,...,False,,,0.0,443.350176,relation/14126301,450.0,1.0,0.977109,LINESTRING (0.7161575436002887 51.440498299145...
relation/15772117-320-DC,GB2 1,FR0 0,,DC,0.970992,True,0,inf,1000.0,0.0,...,False,,,0.0,396.846175,relation/15772117,320.0,1.0,0.714775,LINESTRING (1.1448358124246543 51.098446880403...
relation/15781671-525-DC,GB2 1,DK0 0,,DC,0.960729,True,0,inf,1400.0,0.0,...,False,,,0.0,853.519524,relation/15781671,525.0,1.0,0.816407,LINESTRING (-0.2365005345670103 52.92099253311...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
XK0 0 rural gas boiler,XK0 0 gas,XK0 0 rural heat,,rural gas boiler,0.975000,True,0,20.0,0.0,0.0,...,False,,,NaN,0.000000,,NaN,NaN,NaN,
XK0 0 rural ground heat pump,XK0 0 low voltage,XK0 0 rural heat,,rural ground heat pump,1.000000,True,0,20.0,0.0,0.0,...,False,,,NaN,0.000000,,NaN,NaN,NaN,
XK0 0 urban decentral air heat pump,XK0 0 low voltage,XK0 0 urban decentral heat,,urban decentral air heat pump,1.000000,True,0,18.0,0.0,0.0,...,False,,,NaN,0.000000,,NaN,NaN,NaN,


In [39]:
n1.links.carrier.unique()

array(['DC', 'co2 sequestered', 'OCGT', 'CCGT', 'H2 Electrolysis',
       'gas pipeline', 'gas pipeline new', 'battery charger',
       'battery discharger', 'Sabatier', 'SMR CC', 'SMR', 'BEV charger',
       'V2G', 'oil refining', 'land transport oil',
       'urban central air heat pump', 'urban central gas boiler',
       'urban central gas CHP', 'urban central gas CHP CC',
       'unsustainable bioliquids', 'biogas to gas',
       'urban central solid biomass CHP',
       'urban central solid biomass CHP CC', 'biomass to liquid',
       'electrobiofuels', 'Haber-Bosch', 'ammonia cracker',
       'biomass-to-methanol', 'OCGT methanol',
       'heat100-200 industry solid biomass',
       'heat100-200 industry solid biomass CC',
       'heat100-200 industry industrial heat pump high temperature',
       'heat100-200 industry electric boiler steam',
       'heat200-500 industry solid biomass',
       'heat200-500 industry solid biomass CC',
       'heat200-500 industry hydrogen', 'heat